# Task 2 — four tuned Spark ML classifiers and evaluation

**Run after Task 1 on the actual staged Parquet.** The task is binary classification, not collision-severity regression: compare four classifiers from at least three families, tune all four, score the same untouched Nov–Dec test period and save real Spark PipelineModels. A fresh kernel/session helps ensure your actual cluster configuration is captured.

## 1️⃣ SPARK SESSION CONFIGURATION + TASK 1 GATE

Use the same verified student/allocation, approved resources and Spark version as your assessed Task 1 run. Settings requested inside an existing JVM may not take effect.

In [ ]:
from pathlib import Path
import sys
from pyspark.sql import SparkSession
ROOT = Path.cwd()
if not (ROOT / "coursework").is_dir():
    ROOT = ROOT.parent  # also works when Jupyter starts inside notebooks/
if not (ROOT / "coursework").is_dir():
    raise RuntimeError("Launch Jupyter from the TR-04 project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from coursework.settings import (load_config, make_spark, project_path,
    read_json, require_verified_allocation, spark_configuration)
cfg = load_config()  # gitignored config/config.json: your real allocation and cluster
require_verified_allocation(cfg)  # requires YOUR independently checked Aula row and terms
spark = make_spark(cfg, "Task2")  # config-driven SparkSession.builder, not guessed Colab resources
assert isinstance(spark, SparkSession)
print("Student and allocated dataset:", cfg["student"], cfg["allocation"]["pool_reference"],
      cfg["allocation"]["dataset_name"])
print("ACTUAL Spark application / resources:", spark_configuration(spark))


## 2️⃣ READ STAGED PARQUET + TIME SPLIT

This is the **same** Task 1 output; no sampling of the final test. Use the real observed Task 1 counts rather than rerunning a huge `count()` simply to print an example number.

In [ ]:
from coursework.data import load_processed, split_processed
meta = read_json(project_path(cfg, "results_dir") / "task1.json")
assert meta["status"] == "observed" and all(meta["big_data_verification"].values())
data = load_processed(spark, cfg)
train, october, final_test = split_processed(data)
print("Measured Jan–Sep / October / Nov–Dec:", meta["temporal_split_rows"])
print("Target:", meta["target_definition"])
print("Staged Parquet:", meta["processed_parquet_path"])


## 3️⃣ CLASS BALANCE + DAY-GROUPED CV FOLDS

Use training-only prevalence for inverse class weights on LR/RF/GBT. Spark 3.5 MLP does not support `weightCol`, so it is fitted unweighted on the same Jan–Sep rows and its different loss must be disclosed. Each pickup date belongs to one CV fold, but day-grouped CV is not rolling-origin validation. Thresholds use October only.

In [ ]:
train_cells = [r for r in meta["counts_by_month_and_class"]
               if r["year_month"] < "2019-10"]
by_label = {label: sum(int(x["count"]) for x in train_cells if float(x["label"]) == label)
            for label in (0.0, 1.0)}
assert by_label[0.0] and by_label[1.0]
total_train = sum(by_label.values())
print("Jan–Sep class counts:", by_label)
print("Positive prevalence:", by_label[1.0] / total_train)
print("Training-only inverse class weights:",
      {label: total_train / (2 * n) for label, n in by_label.items()})
print("CV folds / tuning fraction:", cfg["model"]["cv_folds"],
      cfg["model"]["tuning_fraction"])
print("NOTE: Spark MLP has no weightCol; its fit is UNWEIGHTED on these same rows.")


## 4️⃣ VECTOR ASSEMBLY + FOUR MLLIB CLASSIFIER FAMILIES

Fixed pickup-time category vocabularies define the SAME vector width in every fold. Learned `Imputer`/`OneHotEncoder`/`StandardScaler` fit **inside** each CV Pipeline; VectorAssembler excludes late/label fields. Families: LR (linear), RF (tree ensemble), Spark MLP (neural) and GBT (free choice). Do not call LinearSVC a nonlinear kernel model.

In [ ]:
from pyspark.ml.classification import (LogisticRegression, RandomForestClassifier,
    MultilayerPerceptronClassifier, GBTClassifier)
from pyspark.ml.feature import VectorAssembler
from coursework.models import FEATURE_VECTOR_SIZE, make_preprocessing_pipeline, build_model_specs
from coursework.data import FORBIDDEN_AT_PICKUP
prep = make_preprocessing_pipeline()
assembler = next(stage for stage in prep.getStages() if isinstance(stage, VectorAssembler))
assert set(assembler.getInputCols()).isdisjoint(FORBIDDEN_AT_PICKUP)
specs = build_model_specs()
assert len(specs) == 4 and {'linear', 'tree_ensemble', 'neural'} <= {s.family for s in specs}
for s in specs:
    print("MODEL:", s.name, "FAMILY:", s.family, "WHY:", s.rationale_to_check)
print("VectorAssembler inputs (late/outcome columns absent):", assembler.getInputCols())
print("Predeclared constant feature width for Spark neural network:", FEATURE_VECTOR_SIZE)


## 5️⃣ DISTRIBUTED CROSS VALIDATION — EACH MODEL HAS A GRID

`run_task2` constructs this `CrossValidator` blueprint **four times** with each actual param grid, `areaUnderPR` evaluator and day-grouped `foldCol`. The cell creates one *illustrative object only* (no fit); Step 6 does all four real CV fits. The reference project tuned only RF and used regression RMSE, which would not satisfy TR-04.

In [ ]:
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from coursework.models import pipeline_for, param_grid_as_dicts
for s in specs:
    print(s.name, "candidate parameters:", param_grid_as_dicts(s))
first = specs[0]
cv_blueprint = CrossValidator(
    estimator=pipeline_for(first.estimator), estimatorParamMaps=first.param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="label",
        rawPredictionCol="rawPrediction", metricName="areaUnderPR"),
    numFolds=int(cfg["model"]["cv_folds"]), foldCol="cv_fold",
    parallelism=int(cfg["model"]["cv_parallelism"]),
)
print("Illustrative fold-aware CV blueprint (NOT FITTED HERE):",
      cv_blueprint.getNumFolds(), "folds")


## 6️⃣ FIT FOUR CVs + FULL-TRAIN REFITS + REAL MODEL SAVES

**This is the expensive assessed operation**: four CV grids on a documented Jan–Sep training-only sample, then one full-Jan–Sep best-param refit per model. Spark PipelineModels are actually saved to `artifacts/models/`; no arbitrary 50k-row sklearn comparison. Thresholds are chosen on October; Nov–Dec is scored once per fitted model.

In [ ]:
from coursework.task2 import run_task2
observed_2 = run_task2(spark, cfg)  # ACTUAL four CrossValidator fits, refits and holdout scoring
assert observed_2["status"] == "observed" and len(observed_2["models"]) == 4
for row in observed_2["models"]:
    path = project_path(cfg, "artifact_dir") / "models" / row["name"]
    assert path.is_dir(), path
    print(row["name"], "CV folds:", row["cv_folds"],
          "CV seconds:", row["cv_seconds"], "full fit seconds:", row["final_fit_seconds"],
          "weighting:", row["training_weight_policy"], "saved Spark PipelineModel:", path)


## 7️⃣ HOLDOUT METRICS + CONFUSION TABLE

The Spark evaluators compute ROC-AUC/PR-AUC; also show positive-class precision, recall, F1, accuracy and all four confusion cells for **the same Nov–Dec rows**. Always-positive accuracy and PR-AUC baseline should be compared to *measured* test prevalence.

In [ ]:
import pandas as pd
from IPython.display import display
rows = []
for m in observed_2["models"]:
    v = m["metrics"]
    rows.append({"model": m["name"], "family": m["family"],
        "PR-AUC": v["auc_pr"], "ROC-AUC": v["auc_roc"],
        "precision": v["positive_precision"], "recall": v["positive_recall"],
        "F1": v["positive_f1"], "accuracy": v["accuracy"],
        "TN": v["confusion"]["tn"], "FP": v["confusion"]["fp"],
        "FN": v["confusion"]["fn"], "TP": v["confusion"]["tp"],
        "CV_s": m["cv_seconds"], "full_fit_s": m["final_fit_seconds"]})
comparison = pd.DataFrame(rows).set_index("model")
display(comparison)
print("Measured no-skill PR baseline on Nov–Dec = positive prevalence:",
      observed_2["models"][0]["metrics"]["base_rate"])


## 8️⃣ ONE-CELL EP2: FOUR BEST PARAM/TIMES + CONFUSION + ROC/PR

Rebuild a SINGLE composite EP2 image in THIS cell from all four real CV-winning parameter sets and fit times, four confusion matrices and held-out ROC/PR panels. Curves use binned plotting coordinates; numerical AUCs use Spark evaluators.

In [ ]:
from IPython.display import Image, display
from coursework.evaluation import plot_model_evidence
series = read_json(project_path(cfg, "results_dir") / "task2_curves.json")["curves"]
ep2 = project_path(cfg, "results_dir") / "task2_evidence.png"
plot_model_evidence(observed_2["models"], series, ep2)  # one cell -> one EP2 composite
display(Image(filename=str(ep2)))
print("Numerical ROC/PR AUC values come from Spark, not binned plot coordinates.")


## 9️⃣ BEST GRID, OCTOBER THRESHOLD + TREE IMPORTANCE

Show *each candidate grid score* as well as the chosen parameters. Global split importance comes from fitted tree models and is not a causal effect; Task 3's LIME is one **local** explanation. The winning algorithm is selected from CV PR-AUC with a predeclared time tie-breaker, not from Nov–Dec results.

In [ ]:
import matplotlib.pyplot as plt
for m in observed_2["models"]:
    print("MODEL:", m["name"], "grid:", m["param_grid"],
          "CV candidate PR-AUCs:", m["cv_scores"], "best:", m["best_params"],
          "October threshold:", m["threshold_tuning"])
rf = next(m for m in observed_2["models"] if m["name"] == "RandomForestClassifier")
importance = pd.DataFrame(rf["tree_global_split_importance"])
display(importance)
ax = importance.head(8).sort_values("global_split_importance").plot.barh(
    x="feature", y="global_split_importance", legend=False, figsize=(8, 3.5), color="#315c84")
ax.set(title="Fitted RF global split importance (not causal)", xlabel="relative importance")
plt.tight_layout(); plt.show()
print("Predeclared CV-selected model:", observed_2["recommended_model"])


## 🔟 MODEL SERIALIZATION (GENUINE SPARK PIPELINEMODEL)

The reference's 'model serialization' cell trains an unrelated small sklearn model and never saves anything. Here the *real fitted Spark PipelineModel* from Step 6 is reloaded; the pipeline contains preprocessing and the classifier. Do not recalculate/test-tune metrics on the final holdout after inspecting them.

In [ ]:
from pyspark.ml import PipelineModel
best = observed_2["recommended_model"]
model_path = project_path(cfg, "artifact_dir") / "models" / best
reloaded = PipelineModel.load(str(model_path))
print("Reloaded trained Spark model:", best)
print("Saved stages:", [type(stage).__name__ for stage in reloaded.stages])
print("Measured Task 2 JSON:", project_path(cfg, "results_dir") / "task2.json")
print("Your interpretation must include errors, choice rationale and serving limitations.")


**Task 2 completion:** document actual four tuned classifiers, their grids/metrics/fit times and same-cohort comparison. Do not claim a PR-AUC difference is statistically significant without uncertainty analysis; do not claim booking-time predictions for unknown future payment method.